# Module 8 — Mocking (code-along)

**When do you mock?** At the boundary between your code and something you can't run in a
unit test — the network, a paid API, a clock. Cheap in-memory logic you run *for real*.

This notebook demonstrates the **mock mechanics** on tiny inline functions (stdlib
`unittest.mock` only, so every cell runs anywhere). The catalog-specific `TestClient` /
`requests.get` / `get_products` code is shown as illustration — you run *those* for real
in the lab.

## §1 · Run the cheap thing for real (`TestClient`)

The catalog + server are cheap and in-process, so you test them for real — no mock.
In the lab you do this with FastAPI's `TestClient`:

```python
from fastapi.testclient import TestClient
from catalog.server import app
client = TestClient(app)

def test_list_returns_seed():
    r = client.get("/products")
    assert r.status_code == 200
    assert len(r.json()) == 5        # real app, real catalog, no @patch
```

Below: the same idea in miniature — assert on a real object directly, no fakes.

In [ ]:
# a real, cheap object — just assert on it, no mock needed
catalog = {1: {"id": 1, "name": "Cable"}, 2: {"id": 2, "name": "Mat"}}

def get(cid):
    if cid not in catalog:
        raise KeyError(f"no id {cid}")
    return catalog[cid]

assert get(1)["name"] == "Cable"
assert len(catalog) == 2
try:
    get(999)
except KeyError as e:
    print("missing id raised for real:", e)
print("ran the real thing — no mock")

## §2 · Mock the edge — `patch` / `MagicMock`, `return_value`

Now the part you *can't* run for real: a call that leaves the machine. Replace it with a
fake you control. In the lab that call is `requests.get`:

```python
from unittest.mock import patch
from catalog.client import get_products
from catalog.models import Product

def test_returns_typed_products():
    with patch("catalog.client.requests.get") as mock_get:
        mock_get.return_value.json.return_value = [
            {"id": 1, "name": "Cable", "category": "Electronics",
             "price": 499.0, "in_stock": True, "tags": []}]
        mock_get.return_value.raise_for_status.return_value = None
        result = get_products()
    assert isinstance(result[0], Product)
```

Below: the mechanic on an inline `fetch` that takes its network call as an argument.

In [ ]:
from unittest.mock import MagicMock

def fetch(getter, base_url="http://x"):
    resp = getter(f"{base_url}/items")   # the "network call"
    resp.raise_for_status()
    return resp.json()

# build a fake response you fully control
fake = MagicMock()
fake.json.return_value = [{"id": 1, "name": "Cable"}]
fake.raise_for_status.return_value = None

result = fetch(lambda url: fake)
print(result)
assert result[0]["name"] == "Cable"     # got the fake's data — no network hit
print("returned the fake's data, no network")

## §3 · Verify the call, then simulate failure

A mock records *how* it was called (`assert_called_once_with`) and can be made to fail
(`side_effect`). First: assert the exact call.

In [ ]:
getter = MagicMock()
getter.return_value.json.return_value = []
getter.return_value.raise_for_status.return_value = None

fetch(getter, "http://x")
getter.assert_called_once_with("http://x/items")   # right URL, called exactly once
print("verified the exact call:", getter.call_args)

`side_effect` makes the fake **raise** — you can't drop the real network on cue, but a
mock can. In the lab: `mock_get.side_effect = requests.ConnectionError("down")`.

In [ ]:
boom = MagicMock(side_effect=ConnectionError("down"))
try:
    fetch(boom)
    print("no error?!")
except ConnectionError as e:
    print("side_effect raised on cue:", e)

### The test that earns its keep — malformed response → validation error

"The network raised" only re-raises. The *valuable* test: the server returns a **bad row**
and your client refuses to hand it back. In the lab, Pydantic does this:

```python
from pydantic import ValidationError
def test_rejects_malformed_response():
    with patch("catalog.client.requests.get") as mock_get:
        mock_get.return_value.json.return_value = [{"id": 1, "name": "X",
            "category": "c", "price": -5, "in_stock": True, "tags": []}]
        mock_get.return_value.raise_for_status.return_value = None
        with pytest.raises(ValidationError):
            get_products()
```

Below: the same boundary-validation idea with a tiny inline validator.

In [ ]:
def to_product(row):
    if row["price"] < 0:                     # the boundary rule (Pydantic does this for real)
        raise ValueError(f"bad price {row['price']}")
    return row

def fetch_validated(getter):
    resp = getter("http://x/items"); resp.raise_for_status()
    return [to_product(r) for r in resp.json()]

bad = MagicMock()
bad.json.return_value = [{"id": 1, "name": "X", "price": -5}]
bad.raise_for_status.return_value = None
try:
    fetch_validated(lambda u: bad)
except ValueError as e:
    print("malformed row rejected at the boundary:", e)

---
**Now do the lab** — `modules/m08-mocking/lab/README.md`:
`tests/test_server.py` (TestClient, for real) + `tests/test_client.py` (mock the network).
Run `uv run pytest tests/test_server.py tests/test_client.py -q` → **9 passed**.